In [1]:
import pandas as pd
import seaborn as sns
import matplotlib as plt
import numpy as np
import sklearn
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from xgboost import XGBClassifier
df=pd.read_csv('../data/raw/cleaned_data.csv')

In [2]:
df.drop(columns=['workouts_completed'],inplace=True)


C:\Users\lenov\AppData\Local\Temp\ipykernel_27620\3788675037.py:1: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(columns=['workouts_completed'],inplace=True)


In [3]:
df.isnull().sum()

plan_type              0
monthly_price          0
acquisition_channel    0
primary_device         0
age_group              0
churned                0
minutes_active         0
classes_booked         0
support_tickets        0
subscription_month     0
subscription_year      0
signup_year            0
signup_month           0
dtype: int64

In [4]:
df.sample(5)

,plan_type,monthly_price,acquisition_channel,primary_device,age_group,churned,minutes_active,classes_booked,support_tickets,subscription_month,subscription_year,signup_year,signup_month
11517,1,5.0,3,1,0.0,0,0.0,0,0,11,2025,2025,8
15619,0,0.0,4,0,1.0,1,226.0,0,0,12,2024,2024,10
1907,1,1.0,4,1,0.0,1,113.0,0,0,2,2025,2025,1
28029,0,2.0,3,0,1.0,0,94.0,1,0,9,2025,2025,9
28378,1,5.0,0,1,1.0,0,345.0,3,0,12,2025,2025,3


In [5]:
X = df.drop(columns=["churned"])
y = df["churned"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"Train Churn rate: {y_train.mean()*100:.4f}%")
print(f"Test  Churn rate: {y_test.mean()*100:.4f}%")

Train: (24412, 12)  |  Test: (6103, 12)
Train Churn rate: 35.4866%
Test  Churn rate: 35.4907%


In [6]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42, n_estimators=100),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "XGBoost": XGBClassifier(random_state=42, n_estimators=100)}

In [7]:
for name, model in models.items():
    print(f"================ {name} ================")
    
    # Use scaled data for Logistic Regression, tree-based models can use unscaled data
    if name == "Logistic Regression":
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]
        
    # Metrics
    print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
    print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob):.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    print("\n")

================ Logistic Regression ================
Accuracy:  0.7352
ROC-AUC:   0.7905

Classification Report:
              precision    recall  f1-score   support

           0       0.77      0.85      0.81      3937
           1       0.66      0.53      0.59      2166

    accuracy                           0.74      6103
   macro avg       0.71      0.69      0.70      6103
weighted avg       0.73      0.74      0.73      6103



================ Random Forest ================
Accuracy:  0.8070
ROC-AUC:   0.8812

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.88      0.85      3937
           1       0.75      0.68      0.71      2166

    accuracy                           0.81      6103
   macro avg       0.79      0.78      0.78      6103
weighted avg       0.80      0.81      0.80      6103



================ Gradient Boosting ================
Accuracy:  0.7663
ROC-AUC:   0.8347

Classification Report:
        

Since `XGBoost Classifier` has performed best among `Logistic Regression`,`Random Forest` and `XGBoost Classifier`.we will pickle the model and integrate it further with various components to display a interactive dashboard.

In [11]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    roc_curve,
    precision_recall_curve,
    confusion_matrix
)
from xgboost import XGBClassifier

# ---------------------------------------------------------
# 1. Load Data & Validation Split
# ---------------------------------------------------------
DATA_PATH = "../data/raw/cleaned_data.csv"
df = pd.read_csv(DATA_PATH)

# Drop redundant or non-predictive binned columns if present
if "workouts_completed" in df.columns:
    df = df.drop(columns=["workouts_completed"])

X = df.drop(columns=["churned"])
y = df["churned"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Dataset Dimensions: Train = {X_train.shape}, Test = {X_test.shape}")
print(f"Target Distribution: Train = {y_train.mean():.4%}, Test = {y_test.mean():.4%}")

# Stratified 5-Fold Cross Validation Setup
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ---------------------------------------------------------
# 2. Hyperparameter Search Grids
# ---------------------------------------------------------
param_grids = {
    "Random Forest": {
        "model": RandomForestClassifier(random_state=42, n_jobs=-1),
        "params": {
            "n_estimators": [100, 200, 300, 400],
            "max_depth": [6, 8, 10, 12, 15, None],
            "min_samples_split": [2, 5, 10, 15],
            "min_samples_leaf": [1, 2, 4, 8],
            "max_features": ["sqrt", "log2", 0.7, 0.8],
            "criterion": ["gini", "entropy"]
        },
        "n_iter": 25
    },
    "XGBoost": {
        "model": XGBClassifier(
            random_state=42, 
            eval_metric="logloss", 
            use_label_encoder=False, 
            n_jobs=-1
        ),
        "params": {
            "n_estimators": [100, 200, 300, 450],
            "max_depth": [3, 4, 5, 6, 8],
            "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.15],
            "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
            "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
            "min_child_weight": [1, 3, 5, 7],
            "gamma": [0, 0.1, 0.2, 0.4, 0.8],
            "reg_alpha": [0, 0.01, 0.1, 1.0],
            "reg_lambda": [0.5, 1.0, 1.5, 2.0]
        },
        "n_iter": 35
    }
}

# ---------------------------------------------------------
# 3. Randomized Search Cross-Validation Execution
# ---------------------------------------------------------
tuned_results = {}

for model_name, config in param_grids.items():
    print(f"\n================ Tuning {model_name} ================")
    search = RandomizedSearchCV(
        estimator=config["model"],
        param_distributions=config["params"],
        n_iter=config["n_iter"],
        scoring="roc_auc",
        cv=cv_strategy,
        random_state=42,
        n_jobs=-1,
        verbose=1
    )
    
    search.fit(X_train, y_train)
    
    best_estimator = search.best_estimator_
    y_test_probs = best_estimator.predict_proba(X_test)[:, 1]
    y_test_pred = (y_test_probs >= 0.50).astype(int)
    
    test_roc = roc_auc_score(y_test, y_test_probs)
    test_acc = accuracy_score(y_test, y_test_pred)
    test_f1 = f1_score(y_test, y_test_pred)
    test_prec = precision_score(y_test, y_test_pred)
    test_rec = recall_score(y_test, y_test_pred)
    
    tuned_results[model_name] = {
        "best_estimator": best_estimator,
        "best_params": search.best_params_,
        "best_cv_roc_auc": search.best_score_,
        "test_roc_auc": test_roc,
        "test_accuracy": test_acc,
        "test_f1": test_f1,
        "test_precision": test_prec,
        "test_recall": test_rec,
        "y_prob": y_test_probs,
        "y_pred": y_test_pred
    }
    
    print(f"Top CV ROC-AUC: {search.best_score_:.4f}")
    print(f"Test ROC-AUC:   {test_roc:.4f} | Test Accuracy: {test_acc:.4f} | Test F1: {test_f1:.4f}")
    print("Optimal Parameters:")
    for k, v in search.best_params_.items():
        print(f"  - {k}: {v}")

# ---------------------------------------------------------
# 4. Comparative Scorecard & Champion Selection
# ---------------------------------------------------------
benchmark_df = pd.DataFrame({
    model: {
        "CV ROC-AUC": res["best_cv_roc_auc"],
        "Test ROC-AUC": res["test_roc_auc"],
        "Test Accuracy": res["test_accuracy"],
        "Test Precision": res["test_precision"],
        "Test Recall": res["test_recall"],
        "Test F1-Score": res["test_f1"]
    }
    for model, res in tuned_results.items()
}).T

print("\n================ Comparative Benchmark Table ================")
print(benchmark_df.round(4))

# Select the model with the highest Test ROC-AUC
champion_name = benchmark_df["Test ROC-AUC"].astype(float).idxmax()
champion_model = tuned_results[champion_name]["best_estimator"]
champion_probs = tuned_results[champion_name]["y_prob"]

print(f"\nChampion Model Designated: {champion_name}")

# ---------------------------------------------------------
# 5. Threshold Optimization Analysis
# ---------------------------------------------------------
thresholds = np.linspace(0.20, 0.80, 61)
threshold_metrics = []

for t in thresholds:
    preds = (champion_probs >= t).astype(int)
    threshold_metrics.append({
        "threshold": t,
        "accuracy": accuracy_score(y_test, preds),
        "precision": precision_score(y_test, preds, zero_division=0),
        "recall": recall_score(y_test, preds),
        "f1": f1_score(y_test, preds)
    })

thresh_df = pd.DataFrame(threshold_metrics)
best_f1_idx = thresh_df["f1"].idxmax()
optimal_threshold = thresh_df.loc[best_f1_idx, "threshold"]
print(f"\nOptimal Decision Threshold for Max F1: {optimal_threshold:.2f}")
print(thresh_df.loc[best_f1_idx].to_frame().T.to_string(index=False))

# Re-evaluate champion on the tuned decision threshold
optimal_preds = (champion_probs >= optimal_threshold).astype(int)
print(f"\nFinal Classification Report ({champion_name} @ Threshold = {optimal_threshold:.2f}):")
print(classification_report(y_test, optimal_preds, digits=4))

# ---------------------------------------------------------
# 6. Model Serialization & Export
# ---------------------------------------------------------
os.makedirs("../models", exist_ok=True)
export_payload = {
    "model_name": champion_name,
    "model": champion_model,
    "best_params": tuned_results[champion_name]["best_params"],
    "optimal_threshold": float(optimal_threshold),
    "features": X_train.columns.tolist(),
    "training_stats": {
        "roc_auc": float(tuned_results[champion_name]["test_roc_auc"]),
        "accuracy": float(accuracy_score(y_test, optimal_preds)),
        "f1_score": float(f1_score(y_test, optimal_preds))
    }
}

export_path = "../models/best_model.pkl"
with open(export_path, "wb") as f:
    pickle.dump(export_payload, f)

print(f"\nChampion model artifacts successfully exported to: {export_path}")

# Verify clean deployment load
with open(export_path, "rb") as f:
    loaded = pickle.load(f)
print(f"Verified model load: {loaded['model_name']} with {len(loaded['features'])} input features.")

Dataset Dimensions: Train = (24412, 12), Test = (6103, 12)
Target Distribution: Train = 35.4866%, Test = 35.4907%

================ Tuning Random Forest ================
Fitting 5 folds for each of 25 candidates, totalling 125 fits
Top CV ROC-AUC: 0.8725
Test ROC-AUC:   0.8779 | Test Accuracy: 0.8027 | Test F1: 0.7055
Optimal Parameters:
  - n_estimators: 400
  - min_samples_split: 5
  - min_samples_leaf: 1
  - max_features: log2
  - max_depth: 15
  - criterion: gini

================ Tuning XGBoost ================
Fitting 5 folds for each of 35 candidates, totalling 175 fits


c:\Users\lenov\Downloads\customer-churn-analysis-datasets\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [14:02:36] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Top CV ROC-AUC: 0.8871
Test ROC-AUC:   0.9042 | Test Accuracy: 0.8257 | Test F1: 0.7473
Optimal Parameters:
  - subsample: 0.6
  - reg_lambda: 2.0
  - reg_alpha: 0.1
  - n_estimators: 450
  - min_child_weight: 3
  - max_depth: 8
  - learning_rate: 0.15
  - gamma: 0
  - colsample_bytree: 0.7

================ Comparative Benchmark Table ================
               CV ROC-AUC  Test ROC-AUC  Test Accuracy  Test Precision  \
Random Forest      0.8725        0.8779         0.8027          0.7503   
XGBoost            0.8871        0.9042         0.8257          0.7696   

               Test Recall  Test F1-Score  
Random Forest       0.6657         0.7055  
XGBoost             0.7262         0.7473  

Champion Model Designated: XGBoost

Optimal Decision Threshold for Max F1: 0.31
 threshold  accuracy  precision   recall       f1
      0.31  0.816648   0.700191 0.845337 0.765949

Final Classification Report (XGBoost @ Threshold = 0.31):
              precision    recall  f1-score   supp

In [8]:
import pickle
best_model_name = "XGBoost"
best_model_path = '../models/best_model.pkl'
with open(best_model_path, 'wb') as file:
    pickle.dump(models[best_model_name], file)

In [9]:
# Save as processed_data.csv inside data folder
df.to_csv('../data/processed_data.csv', index=False)

In [10]:
df.columns.tolist()

['plan_type',
 'monthly_price',
 'acquisition_channel',
 'primary_device',
 'age_group',
 'churned',
 'minutes_active',
 'classes_booked',
 'support_tickets',
 'subscription_month',
 'subscription_year',
 'signup_year',
 'signup_month']